# GADS Coverage Diagnostic

Investigate why only 8/2,246 GADS events match the intake-checklist authoritative bridge.
The intake checklist's downtime-history sheet contains only 8 GADRS-source rows (all RV2 Boiler);
the other 6 downtime rows have `source_system = AIMM` (a maintenance-work-order system,
not GADS). So the upper bound for authoritative GADS matches is 8.

This notebook answers: **what GADS events exist in our enriched table that we could
reasonably tie to our 3 in-scope assets even though they're not in the intake sheet?**

In [ ]:
from pyspark.sql import functions as F

GADS = spark.table("gads_events_enriched")
print(f"gads_events_enriched: {GADS.count():,} total rows")
print("\nrow count per UNIT_ID:")
GADS.groupBy("UNIT_ID","UNIT_NAME").count().orderBy("UNIT_ID").show(truncate=False)

## 1. Do the 8 intake GEN_SEQ_NOs exist in gads_events_enriched?

In [ ]:
INTAKE_GEN_SEQ = [861409, 861410, 863238, 863239, 863475, 863476, 864423, 864424]

found = (GADS.filter(F.col("GEN_SEQ_NO").isin(*INTAKE_GEN_SEQ))
    .select("GEN_SEQ_NO","UNIT_ID","UNIT_NAME","REAL_START_DT","REAL_END_DT",
            "EVENT_TYPE_DESC","CAUSE_CODE_DESC","EQUIPMENT_DESC","CAUSE_OF_EVENT")
    .orderBy("GEN_SEQ_NO"))
print(f"Matching {found.count()} of {len(INTAKE_GEN_SEQ)} intake-checklist GEN_SEQ_NOs:")
found.show(truncate=False)

## 2. Full RV2 / RV3 event landscape

We have 86=RV2 and 87=RV3. Let's see what's actually in the data.

In [ ]:
only_lg = GADS.filter(F.col("UNIT_ID").isin(86, 87))
print(f"RV2+RV3 events: {only_lg.count():,}")

print("\n[Top 20 EVENT_TYPE_DESC per unit]")
(only_lg.groupBy("UNIT_ID","UNIT_NAME","EVENT_TYPE_DESC").count()
    .orderBy("UNIT_ID", F.col("count").desc()).show(40, truncate=False))

print("[Top 20 CAUSE_CODE_DESC per unit]")
(only_lg.groupBy("UNIT_ID","CAUSE_CODE_DESC").count()
    .orderBy("UNIT_ID", F.col("count").desc()).show(40, truncate=False))

## 3. Per in-scope asset — plausibly related GADS events

We define a free-text search across `EQUIPMENT_DESC + CAUSE_OF_EVENT + EQUIPMENT_DESC + CAUSE_CODE_DESC`
using keywords derived from each in-scope asset's name. This is a *coverage diagnostic only*;
matches are not used to build any fact table.

In [ ]:
def search_count(unit_ids, keywords):
    df = GADS.filter(F.col("UNIT_ID").isin(*unit_ids))
    text = F.upper(F.concat_ws(" | ",
        F.coalesce("EQUIPMENT_DESC", F.lit("")),
        F.coalesce("CAUSE_OF_EVENT", F.lit("")),
        F.coalesce("EVENT_TYPE_DESC", F.lit("")),
        F.coalesce("CAUSE_CODE_DESC", F.lit("")),
    ))
    cond = None
    for kw in keywords:
        c = text.contains(kw.upper())
        cond = c if cond is None else (cond | c)
    return df.filter(cond)

print("=" * 80)
print("RV3 Unit 3 Steam Turbine — plausibly related events")
print("=" * 80)
hits = search_count([87], ["TURBINE", "BLADE", "ROTOR", "GENERATOR", "BEARING", "VIBRATION"])
print(f"Hits: {hits.count():,}")
hits.select("GEN_SEQ_NO","REAL_START_DT","EVENT_TYPE_DESC","CAUSE_CODE_DESC","EQUIPMENT_DESC","CAUSE_OF_EVENT").show(20, truncate=80)

print("=" * 80)
print("RV2 Unit 2 Boiler — plausibly related events")
print("=" * 80)
hits = search_count([86], ["BOILER", "TUBE", "BURNER", "DRUM", "FURNACE", "WATERWALL", "SUPERHEAT", "REHEAT", "FEEDWATER"])
print(f"Hits: {hits.count():,}")
hits.select("GEN_SEQ_NO","REAL_START_DT","EVENT_TYPE_DESC","CAUSE_CODE_DESC","EQUIPMENT_DESC","CAUSE_OF_EVENT").show(20, truncate=80)

print("=" * 80)
print("RV3 East Boiler Feed Pump — plausibly related events")
print("=" * 80)
hits = search_count([87], ["BFP", "FEED PUMP", "FEEDWATER PUMP", "FEED-WATER PUMP", "FEEDPUMP"])
print(f"Hits: {hits.count():,}")
hits.select("GEN_SEQ_NO","REAL_START_DT","EVENT_TYPE_DESC","CAUSE_CODE_DESC","EQUIPMENT_DESC","CAUSE_OF_EVENT").show(20, truncate=80)

## 4. Inferred-bridge coverage by asset

How many GADS events does the existing `dim_gads_event_equipment` (acronym/keyword bridge)
match to the icare_ids that **belong to** the in-scope assets' equipment subtree?
This tells us how much value the inferred match adds even without an authoritative GADRS row.

In [ ]:
# dim_equipment uses 'equipment_name' (not 'asset_name') and exposes pre-computed
# rollup columns 'pi_prefixes_rollup' and 'gads_acronyms_rollup' that already aggregate
# PI prefixes / GADS acronyms down each subtree. We use those rollups directly.
DIM_EQ = spark.table("dim_equipment")
print("dim_equipment columns:", DIM_EQ.columns)

print("\nU2 boiler-ish nodes:")
(DIM_EQ.filter(F.upper(F.col("equipment_name")).contains("BOILER"))
       .filter(F.col("plant") == "RV2")
       .select("icare_id","equipment_name","level","descendant_count",
               "gads_acronyms_rollup","gads_event_count_rollup",
               "pi_prefixes_rollup","pi_tag_count_rollup")
       .orderBy("level","equipment_name").show(30, truncate=60))

print("U3 turbine / generator-ish nodes:")
(DIM_EQ.filter((F.upper(F.col("equipment_name")).contains("TURBINE")) |
              (F.upper(F.col("equipment_name")).contains("GENERATOR")))
       .filter(F.col("plant") == "RV3")
       .select("icare_id","equipment_name","level","descendant_count",
               "gads_acronyms_rollup","gads_event_count_rollup",
               "pi_prefixes_rollup","pi_tag_count_rollup")
       .orderBy("level","equipment_name").show(30, truncate=60))

print("U3 boiler feed pump-ish nodes:")
(DIM_EQ.filter((F.upper(F.col("equipment_name")).contains("FEED PUMP")) |
              (F.upper(F.col("equipment_name")).contains("BFP")) |
              (F.upper(F.col("equipment_name")).contains("FEEDWATER PUMP")))
       .filter(F.col("plant") == "RV3")
       .select("icare_id","equipment_name","level","descendant_count",
               "gads_acronyms_rollup","gads_event_count_rollup",
               "pi_prefixes_rollup","pi_tag_count_rollup")
       .orderBy("level","equipment_name").show(30, truncate=60))

## 5. Summary verdict

The print at the bottom tells you the maximum possible GADS coverage if we add
keyword-based fallbacks for each asset. The intake checklist's GADRS section
is small by design — it lists only the events the owner has already tagged.
We can rescue many more RV2/RV3 events by ranking on the same acronym match
already used in `dim_gads_event_equipment`, plus simple keyword rules
for the 3 in-scope assets.

In [ ]:
bridge = spark.table("dim_gads_event_equipment")
print(f"dim_gads_event_equipment rows: {bridge.count():,}")
print(f"Distinct event_uid covered:    {bridge.select('event_uid').distinct().count():,}")
print(f"Distinct GADS events total:    {GADS.filter(F.col('UNIT_ID').isin(86,87)).count():,}")

print("\nMatched-acronym distribution:")
bridge.groupBy("matched_acronym").count().orderBy(F.col("count").desc()).show(30, truncate=False)